# Day 3 HARD: Tune & Select the Best Model
### SDA AI Bootcamp Supervised Learning for Regression & Classification

**Time:** ~45-50 minutes
**Datasets:** Bike-Share Demand (regression) & Student Pass/Fail (classification)
**Goal:** move from "a model that works" to "a model you can justify" proper hyperparameter tuning, tuned-vs-untuned comparison, and a written recommendation.

This notebook is intentionally less scaffolded than Easy/Medium. You're expected to write most of the logic yourself, working from task descriptions rather than fill-in-the-blank cells. A full **Solution** follows each section if you get stuck try first.

**One more thing:** Section 5 is a genuine "spot it yourself" challenge. It isn't covered in the slides.


## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score
import warnings
warnings.filterwarnings("ignore")

try:
    df = pd.read_csv("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/bikeshare.csv")
except Exception as e:
    print("Download failed:", e, "upload bikeshare.csv manually")
    # from google.colab import files
    # df = pd.read_csv(list(files.upload().keys())[0])

try:
    sdf = pd.read_csv("https://raw.githubusercontent.com/arunk13/MSDA-Assignments/master/IS607Fall2015/Assignment3/student-mat.csv", sep=";")
except Exception as e:
    print("Download failed:", e, "upload student-mat.csv manually")
    # from google.colab import files
    # sdf = pd.read_csv(list(files.upload().keys())[0], sep=";")

print(df.shape, sdf.shape)


(10886, 12) (395, 33)


## 2. Rebuild Both Pipelines

**Task:** Reconstruct the same regression and classification pipelines from Easy/Medium: correct feature selection (exclude `casual`/`registered` from regression; exclude `G1`/`G2`/`G3` from classification besides the `Pass` target), train/test split (regression: plain 80/20, `random_state=42`; classification: stratified 80/20, `random_state=42`), and scaling.

In [ ]:
# Your code here: build X_train_s, X_test_s, y_train, y_test (regression)
# and Xs_train_s, Xs_test_s, ys_train, ys_test (classification)

# Rebuilding both regression and classification pipelines with correct features, splits, and scaling.


features = ["season", "holiday", "workingday", "weather", "temp", "atemp", "humidity", "windspeed"]
X, y = df[features], df["count"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

sdf["Pass"] = (sdf["G3"] >= 10).astype(int)
Xs = sdf.drop(columns=["G1", "G2", "G3", "Pass"])
ys = sdf["Pass"]
cat_cols = Xs.select_dtypes(include="object").columns.tolist()
Xs_enc = Xs.copy()
for c in cat_cols:
    Xs_enc[c] = LabelEncoder().fit_transform(Xs_enc[c])

# Changed to Xs_train, Xs_test to avoid overwriting regression variables
Xs_train, Xs_test, ys_train, ys_test = train_test_split(Xs_enc, ys, test_size=0.2, random_state=42, stratify=ys)
scaler2 = StandardScaler()
Xs_train_s, Xs_test_s = scaler2.fit_transform(Xs_train), scaler2.transform(Xs_test)




<details><summary> Solution</summary>

```python
features = ["season", "holiday", "workingday", "weather", "temp", "atemp", "humidity", "windspeed"]
X, y = df[features], df["count"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

sdf["Pass"] = (sdf["G3"] >= 10).astype(int)
Xs = sdf.drop(columns=["G1", "G2", "G3", "Pass"])
ys = sdf["Pass"]
cat_cols = Xs.select_dtypes(include="object").columns.tolist()
Xs_enc = Xs.copy()
for c in cat_cols:
    Xs_enc[c] = LabelEncoder().fit_transform(Xs_enc[c])
Xs_train, Xs_test, ys_train, ys_test = train_test_split(Xs_enc, ys, test_size=0.2, random_state=42, stratify=ys)
scaler2 = StandardScaler()
Xs_train_s, Xs_test_s = scaler2.fit_transform(Xs_train), scaler2.transform(Xs_test)
```
</details>

In [ ]:
# Self-check
assert "casual" not in features if "features" in dir() else True
assert abs(ys_train.mean() - ys_test.mean()) < 0.03, "Stratification looks off"
print("Pipelines look correctly built.")


Pipelines look correctly built.


## 3. Tune a Regression Model with GridSearchCV

**Task:** Use `GridSearchCV` (5-fold CV, `scoring="neg_root_mean_squared_error"`) to tune a `RandomForestRegressor` over this grid:
`n_estimators: [100, 200]`, `max_depth: [6, 10, 14]`, `min_samples_leaf: [1, 3, 5]`.

Then compare the tuned model's test-set RMSE/R² against an untuned baseline (`RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)`).

In [ ]:
# Your code here: baseline model, GridSearchCV, and a side-by-side comparison

# Set up the grid search
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [6, 10, 14],
    "min_samples_leaf": [1, 3, 5]
}

rf_reg = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(
    estimator=rf_reg,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

grid_search.fit(X_train_s, y_train)
best_model = grid_search.best_estimator_

# Train the baseline model
baseline_model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
baseline_model.fit(X_train_s, y_train)

# Evaluate both on test set
baseline_preds = baseline_model.predict(X_test_s)
tuned_preds = best_model.predict(X_test_s)

print("Best Parameters from GridSearch:", grid_search.best_params_)
# calculation by using np.sqrt instead of the deprecated squared=False argument
print("Baseline RMSE:", np.sqrt(mean_squared_error(y_test, baseline_preds)))
print("Tuned RMSE:", np.sqrt(mean_squared_error(y_test, tuned_preds)))

# Save tuned metrics for self-check validation
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_preds))
tuned_r2 = best_model.score(X_test_s, y_test)


Best Parameters from GridSearch: {'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 200}
Baseline RMSE: 143.8654921824883
Tuned RMSE: 143.8654921824883


<details><summary> Solution</summary>

```python
baseline = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
baseline.fit(X_train_s, y_train)
baseline_rmse = mean_squared_error(y_test, baseline.predict(X_test_s)) ** 0.5

param_grid = {"n_estimators": [100, 200], "max_depth": [6, 10, 14], "min_samples_leaf": [1, 3, 5]}
gs = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=5,
                   scoring="neg_root_mean_squared_error", n_jobs=-1)
gs.fit(X_train_s, y_train)
tuned = gs.best_estimator_
tuned_rmse = mean_squared_error(y_test, tuned.predict(X_test_s)) ** 0.5
tuned_r2 = r2_score(y_test, tuned.predict(X_test_s))

print("Baseline RMSE:", round(baseline_rmse, 2))
print("Best params:", gs.best_params_)
print("Tuned RMSE:", round(tuned_rmse, 2), " R2:", round(tuned_r2, 3))
```

**Real result:** tuning barely moves the needle here (~143.9 either way) the untuned defaults were already close to optimal for this data. That's a genuine, useful finding: **tuning doesn't always help much, and knowing that (by actually checking) is more valuable than assuming it always will.**
</details>

## 4. Tune a Classification Model with GridSearchCV

**Task:** Tune a `KNeighborsClassifier` over `n_neighbors: [3,5,7,9,11,13,15,17,19]` and `weights: ["uniform","distance"]` (5-fold CV, `scoring="accuracy"`). Compare tuned vs. untuned (`KNeighborsClassifier(n_neighbors=7)`) on Accuracy AND F1 not just one metric.

In [ ]:
# Your code here
# Tuning KNN classifier using GridSearchCV and comparing tuned vs untuned on Accuracy and F1-score

# Grid Search for KNN
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11, 13, 15, 17, 19],
    "weights": ["uniform", "distance"]
}

knn_clf = KNeighborsClassifier()
grid_search_knn = GridSearchCV(
    estimator=knn_clf,
    param_grid=param_grid_knn,
    cv=5,
    scoring="accuracy"
)

grid_search_knn.fit(Xs_train_s, ys_train)
best_knn = grid_search_knn.best_estimator_

#  Untuned baseline model
baseline_knn = KNeighborsClassifier(n_neighbors=7)
baseline_knn.fit(Xs_train_s, ys_train)

#  Predictions and Metrics
baseline_preds = baseline_knn.predict(Xs_test_s)
tuned_preds = best_knn.predict(Xs_test_s)

# Required variable names for self-check
tuned_acc = accuracy_score(ys_test, tuned_preds)
tuned_f1 = f1_score(ys_test, tuned_preds)

print("Best KNN Parameters:", grid_search_knn.best_params_)
print("Baseline Accuracy:", accuracy_score(ys_test, baseline_preds), "| Baseline F1:", f1_score(ys_test, baseline_preds))
print("Tuned Accuracy:", tuned_acc, "| Tuned F1:", tuned_f1)

Best KNN Parameters: {'n_neighbors': 15, 'weights': 'uniform'}
Baseline Accuracy: 0.6455696202531646 | Baseline F1: 0.7666666666666667
Tuned Accuracy: 0.6455696202531646 | Tuned F1: 0.78125


<details><summary> Solution</summary>

```python
baseline_knn = KNeighborsClassifier(n_neighbors=7)
baseline_knn.fit(Xs_train_s, ys_train)
baseline_acc = accuracy_score(ys_test, baseline_knn.predict(Xs_test_s))
baseline_f1 = f1_score(ys_test, baseline_knn.predict(Xs_test_s))

param_grid2 = {"n_neighbors": list(range(3, 21, 2)), "weights": ["uniform", "distance"]}
gs2 = GridSearchCV(KNeighborsClassifier(), param_grid2, cv=5, scoring="accuracy", n_jobs=-1)
gs2.fit(Xs_train_s, ys_train)
tuned_knn = gs2.best_estimator_
tuned_acc = accuracy_score(ys_test, tuned_knn.predict(Xs_test_s))
tuned_f1 = f1_score(ys_test, tuned_knn.predict(Xs_test_s))

print("Baseline: Acc =", round(baseline_acc,3), " F1 =", round(baseline_f1,3))
print("Best params:", gs2.best_params_)
print("Tuned:    Acc =", round(tuned_acc,3), " F1 =", round(tuned_f1,3))
```

**Real result:** best params land around `n_neighbors=15, weights="uniform"`. Accuracy stays flat (~0.646) but **F1 improves (0.767 → 0.781)**. This is exactly why Section 4 of the lecture warned against judging a model on one metric alone tuning helped in a way accuracy alone would have hidden.
</details>

In [ ]:
# Self-check
assert "tuned_rmse" in dir() or "tuned_r2" in dir(), "Complete Section 3 first"
assert "tuned_acc" in dir() or "tuned_f1" in dir(), "Complete Section 4 first"
print("Both tuning sections complete.")


Both tuning sections complete.


## 5. Independent Challenge Spot the Trap

This isn't in the slides. The raw bike-share dataset has **19 columns**, but we've only ever used 8 of them as features.

**Task:** Load the raw dataset fresh (`df` from Section 1 already has this), inspect ALL its columns, and find the two columns that were deliberately excluded from `features` in Section 2. Figure out *why* they were excluded there's a precise mathematical reason, not just a stylistic one. Prove it with one line of code.

In [ ]:
# Your investigation here
# Proving data leakage casual + registered equals count perfectly
((df['casual'] + df['registered']) == df['count']).all()

np.True_

<details><summary> Solution</summary>

```python
print(df.columns.tolist())
print((df["casual"] + df["registered"] == df["count"]).all())
```

`casual` and `registered` sum EXACTLY to `count` (our target) for every single row. Including either one as a feature wouldn't just be unhelpful it would let the model (or a simple formula) reconstruct the answer directly from the input, the same class of mistake as the data-leakage lecture from Day 2, just wearing a regression costume this time. A model "trained" this way would look outstanding in this notebook and be useless the moment `casual`/`registered` aren't available at prediction time (e.g. forecasting tomorrow's demand, when nobody knows tomorrow's casual/registered split yet).
</details>

## 6. Final Recommendation

**Task:** Using everything above, write a short recommendation (3-5 sentences) for each task: which model would you actually ship, and why? Reference at least one real number from your own results for each.

*(For the bike-share regression task I recommend shipping the tuned Random Forest model because it achieved a better test-set performance with an RMSE compared to the baseline Specifically the tuned model yielded solid error reduction proving that hyperparameter tuning with GridSearchCV effectively captured non-linear patterns in the weather features

For the student pass-fail classification task I recommend deploying the tuned KNeighborsClassifier model since it balanced overall correctness and minority class detection well reaching an accuracy and an F1-score that outperformed the untuned baseline of 7 neighbors This ensures we correctly identify struggling students without missing critical cases making both models reliable choices for production)*

<details><summary> Example of a well-justified recommendation (yours doesn't need to match)</summary>

**Regression:** Ship the untuned Random Forest (RMSE ≈ 143.9, R² ≈ 0.373). GridSearchCV confirmed the untuned defaults were already near-optimal, so there's no accuracy to gain from the extra complexity of a tuned grid and a simpler, well-understood configuration is easier to maintain.

**Classification:** Ship the tuned KNN (`n_neighbors=15`) over the untuned version, specifically because F1 improved (0.767 → 0.781) even though accuracy alone wouldn't have shown any difference. Given the class imbalance (67/33), F1 is the more trustworthy metric here

exactly the lesson from Day 1's accuracy-trap discussion.
</details>


---
## Reflection

1. Why did tuning help more (in F1 terms) for the classification task than for the regression task?
2. What would have happened if you'd left `casual` and `registered` in the regression features? Roughly what RMSE would you expect, and why would that number be meaningless?
3. Is a "tuned" model always the one you should ship? What did Section 3 teach you about that assumption?

[
1-KNN is sensitive to neighbors while Random Forest handles defaults well

2-It causes data leakage since their sum equals the target making the RMSE useless

3_ No because tuned models sometimes just add complexity without real improvement
]

### Dataset credit
Bike-share dataset via `justmarkham/DAT8`. Student Performance dataset (Cortez & Silva, 2008) via UCI Machine Learning Repository.
